# CineMatch — Letterboxd User Data Scraper (v2: 10k scale)

**Owner:** Geoff (CineMatch)  
**Goal:** Build the user-item rating matrix that drives the recommender system.

## Outputs
- `data/users.csv` — one row per user (username, name, location, total films rated)
- `data/user_ratings.csv` — long format, one row per `(username, film_slug, rating)`
- `cache_users/` — every fetched HTML page (so re-runs are free)

## Plan for 10,000 users
1. **Collect usernames from 4 feeds** — popular reviewers, popular members, popular this-year, popular this-month. Dedup gets us to ~10–12k unique active users.
2. **Per-user scrape** — profile page + up to N ratings pages (configurable). At the default cap of 5 pages we get the user's ~125 most recent rated films.
3. **Resume-safe** — if the kernel crashes mid-run, just run cells §7–§8 again. Already-scraped users are skipped automatically.

## Time/disk reality
- ~22 hours wall-clock for the full 10k at 1.5s/request, 5 pages/user.
- ~1.5–2 GB cache disk usage.
- **Recommended: do a 200-user trial run first** (set `MAX_USERS = 200`, ~45 min) to confirm real data is flowing before you commit your laptop to a multi-day scrape.


## §1. Install dependencies

In [ ]:
%pip install --quiet requests beautifulsoup4 pandas

## §2. Imports & config
**The two big knobs:** `MAX_USERS` and `MAX_RATING_PAGES_PER_USER`.

Start with `MAX_USERS = 200` for your first run. After confirming data looks right, bump to 10000.

In [ ]:
import json
import re
import time
import csv
import hashlib
import random
from datetime import datetime, timezone, timedelta
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://letterboxd.com"

# ---- Tune these -------------------------------------------------------------
MAX_USERS                  = 10000
MAX_RATING_PAGES_PER_USER  = 5      # 5 pages ≈ 125 most-recent rated films per user
DELAY_SECONDS              = 1.5    # politeness floor
ADAPTIVE_BACKOFF_BASE      = 30     # on 429, sleep this many seconds (then doubled per consecutive 429)

# ---- Username sources -------------------------------------------------------
# Each feed has up to 128 pages. We walk all of them and dedupe.
USERNAME_SOURCES = [
    ("reviewers_alltime", "https://letterboxd.com/reviewers/popular/this/all-time/", 128),
    ("members_alltime",   "https://letterboxd.com/members/popular/this/all-time/",   128),
    ("members_year",      "https://letterboxd.com/members/popular/this/year/",       128),
    ("members_month",     "https://letterboxd.com/members/popular/this/month/",      128),
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

OUTPUT_DIR  = Path("data");         OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR   = Path("cache_users");  CACHE_DIR.mkdir(exist_ok=True)
DEBUG_DIR   = Path("debug");        DEBUG_DIR.mkdir(exist_ok=True)
USERS_CSV   = OUTPUT_DIR / "users.csv"
RATINGS_CSV = OUTPUT_DIR / "user_ratings.csv"
USERNAMES_CSV = OUTPUT_DIR / "usernames.csv"

session = requests.Session()
session.headers.update(HEADERS)

print(f"MAX_USERS                 = {MAX_USERS}")
print(f"MAX_RATING_PAGES_PER_USER = {MAX_RATING_PAGES_PER_USER}")
print(f"Username sources:         {len(USERNAME_SOURCES)}")
print(f"Outputs:                  {USERS_CSV}, {RATINGS_CSV}")
print(f"Cache:                    {CACHE_DIR}/")


## §3. HTTP helper
Caching + adaptive backoff. On HTTP 429 (rate-limited) we sleep increasingly long, then resume.

In [ ]:
_consecutive_429 = 0


def cache_path_for(url):
    h = hashlib.md5(url.encode("utf-8")).hexdigest()
    return CACHE_DIR / f"{h}.html"


def fetch(url, retries=3, use_cache=True):
    """GET with politeness delay + retries + adaptive 429 backoff."""
    global _consecutive_429
    cp = cache_path_for(url)
    if use_cache and cp.exists() and cp.stat().st_size > 0:
        return BeautifulSoup(cp.read_text(encoding="utf-8"), "html.parser")
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=30)
            if r.status_code == 200:
                cp.write_text(r.text, encoding="utf-8")
                _consecutive_429 = 0
                time.sleep(DELAY_SECONDS)
                return BeautifulSoup(r.text, "html.parser")
            if r.status_code == 404:
                return None
            if r.status_code == 429:
                _consecutive_429 += 1
                wait = ADAPTIVE_BACKOFF_BASE * (2 ** (_consecutive_429 - 1))
                wait = min(wait, 600)  # cap at 10 minutes
                print(f"  [429] rate-limited. sleeping {wait}s (consecutive: {_consecutive_429})")
                time.sleep(wait)
                continue
            print(f"  [{r.status_code}] {url}  (attempt {attempt + 1})")
        except requests.RequestException as exc:
            print(f"  [err] {url} -- {exc}")
        time.sleep(DELAY_SECONDS * (attempt + 2))
    return None


## §4. Collect usernames from all 4 feeds
First run: ~10 minutes (512 list pages × 1.5s). Cached afterward, so re-runs are instant.

Output: `data/usernames.csv` — one username per line.

In [ ]:
def extract_usernames_from_page(soup):
    """Try multiple strategies to pull usernames from a popular-people list page."""
    out = []

    # A: legacy person-table layout
    for row in soup.select("table.person-table tbody tr"):
        a = row.select_one("h3.title-3 a[href]") or row.select_one("a[href^='/']")
        if a:
            href = a.get("href", "").strip("/")
            if href and "/" not in href:
                out.append(href)

    # B: avatar links
    if not out:
        for a in soup.select("a.avatar[href^='/']"):
            href = a.get("href", "").strip("/")
            if href and "/" not in href:
                out.append(href)

    # C: any anchor whose href is exactly /<slug>/  (with reserved-word filter)
    if not out:
        reserved = {"films", "lists", "members", "reviewers", "search", "about", "pro",
                    "settings", "create-account", "sign-in", "year-in-review", "almanac",
                    "journal", "contact", "terms", "privacy", "official", "showdown"}
        for a in soup.select("a[href]"):
            m = re.match(r"^/([A-Za-z0-9_]+)/?$", a.get("href", ""))
            if m and m.group(1).lower() not in reserved:
                out.append(m.group(1))

    return out


def collect_usernames(sources=USERNAME_SOURCES, target=None, verbose=True):
    """Walk all source feeds, collect deduped usernames. Stops early if target reached."""
    seen = set()
    ordered = []
    for label, base, max_pages in sources:
        if verbose:
            print(f"\n=== Source: {label} ===")
        for pg in range(1, max_pages + 1):
            url = base if pg == 1 else base.rstrip("/") + f"/page/{pg}/"
            soup = fetch(url)
            if soup is None:
                if verbose:
                    print(f"   page {pg}: fetch failed, moving on")
                break
            page_users = extract_usernames_from_page(soup)
            new = 0
            for u in page_users:
                if u not in seen:
                    seen.add(u); ordered.append(u); new += 1
            if verbose and (pg <= 3 or pg % 10 == 0 or new == 0):
                print(f"   page {pg:3d}: +{new} new ({len(ordered)} total)")
            if new == 0:
                # whole page produced no new users -> probably end of feed
                if verbose:
                    print(f"   page {pg}: no new users, stopping this source")
                break
            if target is not None and len(ordered) >= target:
                if verbose:
                    print(f"   reached target {target}, stopping early")
                return ordered
    return ordered


# We collect a bit more than MAX_USERS to give us margin (some users have no ratings)
target_count = int(MAX_USERS * 1.3) if MAX_USERS else None
all_usernames = collect_usernames(target=target_count)
print(f"\nTotal unique usernames collected: {len(all_usernames)}")

# Persist for resume / inspection
pd.DataFrame({"username": all_usernames}).to_csv(USERNAMES_CSV, index=False)
print(f"Saved to {USERNAMES_CSV}")
print("First 10:", all_usernames[:10])


## §5. Per-user parsers
Letterboxd encodes ratings as CSS class `rated-N` where `N` ∈ {1..10} → stars = N/2.

In [ ]:
def parse_profile(soup, username):
    """Pull display name + location from a profile page."""
    name = username
    name_el = soup.select_one("h1.title-1, div.profile-name-wrap h1, h1.primaryname, h1")
    if name_el:
        candidate = name_el.get_text(strip=True)
        # Filter out the site logo "Letterboxd" / "Letterboxd — Your life in film"
        if candidate and not candidate.lower().startswith("letterboxd"):
            name = candidate

    location = None
    for span in soup.select("div.profile-metadata span.label, .metadatum span.label, .metadatum .label"):
        t = span.get_text(strip=True)
        if t and "http" not in t.lower():
            location = t
            break

    return {"username": username, "name": name, "location": location}


_RATED_RE = re.compile(r"rated-(\d+)")


def parse_ratings_page(soup):
    """Return list of (slug, rating) pairs from a single ratings page."""
    pairs = []
    items = soup.select("ul.poster-list li, li.poster-container, li.griditem, ul li.poster-container")
    if not items:
        # Fallback: any <li> with a film-poster div
        items = [li for li in soup.find_all("li") if li.select_one("[data-film-slug], a[href^='/film/']")]
    for li in items:
        # Slug
        slug = None
        poster = li.select_one("[data-film-slug]")
        if poster:
            slug = poster.get("data-film-slug")
        if not slug:
            tl = li.select_one("[data-target-link]")
            if tl:
                m = re.match(r"/film/([^/]+)/?", tl.get("data-target-link", ""))
                if m:
                    slug = m.group(1)
        if not slug:
            a = li.select_one("a[href^='/film/']")
            if a:
                m = re.match(r"/film/([^/]+)/?", a.get("href", ""))
                if m:
                    slug = m.group(1)
        if not slug:
            continue

        # Rating: look for any span with class "rated-N"
        rating = None
        for el in li.find_all(True):
            classes = el.get("class") or []
            for c in classes:
                m = _RATED_RE.match(c)
                if m:
                    try:
                        rating = int(m.group(1)) / 2.0
                    except ValueError:
                        pass
                    break
            if rating is not None:
                break

        if rating is not None:
            pairs.append((slug, rating))
    return pairs


def get_total_ratings_pages(soup):
    pagin = soup.select_one("div.paginate-pages")
    if not pagin:
        return 1
    nums = []
    for a in pagin.select("li a"):
        try:
            nums.append(int(a.get_text(strip=True)))
        except ValueError:
            continue
    return max(nums) if nums else 1


def scrape_user(username, max_pages=MAX_RATING_PAGES_PER_USER, verbose=False):
    profile_soup = fetch(f"{BASE_URL}/{username}/")
    if profile_soup is None:
        return None, []
    profile = parse_profile(profile_soup, username)

    ratings_url = f"{BASE_URL}/{username}/films/ratings/"
    pg1 = fetch(ratings_url)
    if pg1 is None:
        return profile, []

    total_pages = min(get_total_ratings_pages(pg1), max_pages)
    pairs = parse_ratings_page(pg1)
    for pg in range(2, total_pages + 1):
        soup = fetch(f"{ratings_url}page/{pg}/")
        if soup is None:
            break
        pairs.extend(parse_ratings_page(soup))
    return profile, pairs


## §6. Smoke test — scrape ONE user and verify ratings come through

In [ ]:
test_user = all_usernames[0]
print(f"Scraping: {test_user}\n")
profile, pairs = scrape_user(test_user, verbose=True)
print("Profile:", profile)
print(f"Ratings collected: {len(pairs)}")
print("\nFirst 10 (slug, rating):")
for slug, r in pairs[:10]:
    print(f"  {slug:50s}  {r}")

if len(pairs) == 0:
    print("\n⚠️  No ratings parsed. This user may have no rated films, or selectors need updating.")
    print("   Try a different test_user, or run §10 DIAGNOSTIC.")


## §7. Full run — scrape all users (resume-safe)
Reads existing CSVs, skips already-scraped usernames. Checkpoints every 25 users.

**Live progress includes ETA.** If you have to stop, just re-run this cell — it picks up where it left off.

In [ ]:
def load_existing_progress():
    """Return (set_of_done_usernames, list_of_user_rows, list_of_rating_rows)."""
    done = set()
    user_rows = []
    rating_rows = []
    if USERS_CSV.exists():
        existing_users = pd.read_csv(USERS_CSV)
        done.update(existing_users["username"].astype(str).tolist())
        user_rows = existing_users.to_dict("records")
        print(f"Resuming: {len(done)} users already in {USERS_CSV.name}")
    if RATINGS_CSV.exists():
        existing_ratings = pd.read_csv(RATINGS_CSV)
        rating_rows = existing_ratings.to_dict("records")
        print(f"Resuming: {len(rating_rows)} ratings already in {RATINGS_CSV.name}")
    return done, user_rows, rating_rows


def write_outputs(user_rows, rating_rows):
    pd.DataFrame(user_rows).to_csv(USERS_CSV, index=False, quoting=csv.QUOTE_MINIMAL)
    pd.DataFrame(rating_rows, columns=["username", "film_slug", "rating"]).to_csv(
        RATINGS_CSV, index=False, quoting=csv.QUOTE_MINIMAL
    )


def fmt_eta(seconds):
    return str(timedelta(seconds=int(seconds)))


# ----- run -----
done, user_rows, rating_rows = load_existing_progress()

target = all_usernames[:MAX_USERS] if MAX_USERS else all_usernames
todo = [u for u in target if u not in done]
print(f"\nTarget: {len(target)} users. Already done: {len(done)}. To scrape now: {len(todo)}\n")

start_time = time.time()
for i, username in enumerate(todo, start=1):
    profile, pairs = scrape_user(username)
    if profile is None:
        # Profile fetch failed -- record nothing, will retry next run
        continue

    profile["total_films_rated"] = len(pairs)
    profile["date_scraped"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    user_rows.append(profile)
    for slug, rating in pairs:
        rating_rows.append({"username": username, "film_slug": slug, "rating": rating})

    # Progress + ETA
    elapsed = time.time() - start_time
    rate = i / elapsed if elapsed > 0 else 0
    remaining = (len(todo) - i) / rate if rate > 0 else 0
    print(f"[{i}/{len(todo)}] {username:25s}  ratings={len(pairs):4d}  "
          f"elapsed={fmt_eta(elapsed)}  ETA={fmt_eta(remaining)}")

    if i % 25 == 0:
        write_outputs(user_rows, rating_rows)
        print(f"   [checkpoint] {len(user_rows)} users / {len(rating_rows)} ratings written")

write_outputs(user_rows, rating_rows)
print(f"\nDone. {len(user_rows)} users / {len(rating_rows)} ratings written.")


## §8. Inspect the dataset

In [ ]:
users_df = pd.read_csv(USERS_CSV)
ratings_df = pd.read_csv(RATINGS_CSV)

print(f"users.csv:        {users_df.shape}")
print(f"user_ratings.csv: {ratings_df.shape}")
print(f"\nUnique users:     {ratings_df['username'].nunique()}")
print(f"Unique films:     {ratings_df['film_slug'].nunique()}")

print("\nRatings per user (top 10):")
print(ratings_df.groupby("username").size().sort_values(ascending=False).head(10))

print("\nMost-rated films (top 20):")
print(ratings_df.groupby("film_slug").size().sort_values(ascending=False).head(20))

print("\nRating value distribution:")
print(ratings_df["rating"].value_counts().sort_index())

print("\nUsers with 0 ratings:")
print((users_df["total_films_rated"] == 0).sum())


## §9. (Optional) Pivot into wide user-item matrix
Warning: at 10k users × ~50k unique films this is huge. Use sparse for ML.

In [ ]:
# Cap the pivot to most-rated films for sanity
top_films = ratings_df.groupby("film_slug").size().nlargest(500).index
matrix = (ratings_df[ratings_df["film_slug"].isin(top_films)]
            .pivot_table(index="username", columns="film_slug", values="rating"))
print(f"Matrix shape (top 500 films): {matrix.shape}")
print(f"Density: {matrix.notna().sum().sum() / matrix.size:.2%}")
matrix.iloc[:5, :8]


## §10. DIAGNOSTIC — run if smoke test (§6) returned 0 ratings
Dumps a user's ratings page so we can see what's actually there.

In [ ]:
diag_user = test_user  # change this if you want to inspect another user
diag_url = f"{BASE_URL}/{diag_user}/films/ratings/"
print(f"Inspecting: {diag_url}\n")

import requests as _req
r = _req.get(diag_url, headers=HEADERS, timeout=30)
print(f"HTTP status: {r.status_code}")
print(f"Body length: {len(r.text)}")

(DEBUG_DIR / "user_ratings_page.html").write_text(r.text, encoding="utf-8")
print(f"Saved HTML to: {(DEBUG_DIR / 'user_ratings_page.html').resolve()}")

soup = BeautifulSoup(r.text, "html.parser")

print("\nSelector counts:")
for sel in [
    "ul.poster-list li",
    "li.poster-container",
    "li.griditem",
    "[data-film-slug]",
    "a[href^='/film/']",
    "[class*='rated-']",
]:
    print(f"  {sel:35s} -> {len(soup.select(sel))}")

print("\nFirst 5 elements with rated-N class:")
import re as _re
for el in soup.find_all(True):
    classes = el.get("class") or []
    if any(_re.match(r"rated-\d+", c) for c in classes):
        print(f"  {el.name}.{'.'.join(classes)}")
        break
